# MCP First Slice Showcase

This notebook showcases ORBIT's first MCP implementation slice after pre-MCP governance closure.

It demonstrates:
- real stdio MCP server registration
- official Python `mcp` SDK-backed tool listing and tool calling
- MCP governance overlay for real filesystem tools
- governed runtime path examples through `SessionManager`


In [1]:
import sys
from pathlib import Path

ROOT = Path('/Volumes/2TB/MAS/openclaw-core/ORBIT').resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from orbit.runtime.core.session_manager import SessionManager
from orbit.runtime.execution.contracts.plans import ExecutionPlan, ToolRequest
from orbit.runtime.mcp import McpStdioServerConfig, bootstrap_stdio_mcp_server, build_mcp_client, async_register_mcp_server_tools, register_mcp_server_tools
from orbit.store.sqlite_store import SQLiteStore
from orbit.tools.registry import ToolRegistry


## 1. Bootstrap a real stdio MCP filesystem server


In [2]:
bootstrap = bootstrap_stdio_mcp_server(
    McpStdioServerConfig(
        name='filesystem',
        command='npx',
        args=['-y', '@modelcontextprotocol/server-filesystem', str(ROOT)],
    )
)
bootstrap

McpClientBootstrap(server_name='filesystem', normalized_name='filesystem', tool_prefix='mcp__filesystem__', transport='stdio', command='npx', args=['-y', '@modelcontextprotocol/server-filesystem', '/Volumes/2TB/MAS/openclaw-core/ORBIT'], env={})

## 2. Official Python `mcp` SDK-backed tool listing


In [3]:
client = build_mcp_client(bootstrap)
tools = await client.async_list_tools()
[(t.original_name, t.orbit_tool_name, t.description) for t in tools[:15]]

[('read_file',
  'mcp__filesystem__read_file',
  'Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.'),
 ('read_text_file',
  'mcp__filesystem__read_text_file',
  "Read the complete contents of a file from the file system as text. Handles various text encodings and provides detailed error messages if the file cannot be read. Use this tool when you need to examine the contents of a single file. Use the 'head' parameter to read only the first N lines of a file, or the 'tail' parameter to read only the last N lines of a file. Operates on the file as text regardless of extension. Only works within allowed directories."),
 ('read_media_file',
  'mcp__filesystem__read_media_file',
  'Read an image or audio file. Returns the base64 encoded data and MIME type. Only works within allowed directories.'),
 ('read_multiple_files',
  'mcp__filesystem__read_multiple_files',
  "Read the contents of multiple files simultaneously. This is more efficient than reading fi

## 3. Register real MCP tools into ORBIT ToolRegistry


In [4]:
registry = ToolRegistry(ROOT)
registered = await async_register_mcp_server_tools(registry=registry, bootstrap=bootstrap)
{'registered_count': len(registered), 'first_15': registered[:15], 'registry_first_20': registry.list_names()[:20]}

{'registered_count': 14,
 'first_15': ['mcp__filesystem__read_file',
  'mcp__filesystem__read_text_file',
  'mcp__filesystem__read_media_file',
  'mcp__filesystem__read_multiple_files',
  'mcp__filesystem__write_file',
  'mcp__filesystem__edit_file',
  'mcp__filesystem__create_directory',
  'mcp__filesystem__list_directory',
  'mcp__filesystem__list_directory_with_sizes',
  'mcp__filesystem__directory_tree',
  'mcp__filesystem__move_file',
  'mcp__filesystem__search_files',
  'mcp__filesystem__get_file_info',
  'mcp__filesystem__list_allowed_directories'],
 'registry_first_20': ['mcp__filesystem__create_directory',
  'mcp__filesystem__directory_tree',
  'mcp__filesystem__edit_file',
  'mcp__filesystem__get_file_info',
  'mcp__filesystem__list_allowed_directories',
  'mcp__filesystem__list_directory',
  'mcp__filesystem__list_directory_with_sizes',
  'mcp__filesystem__move_file',
  'mcp__filesystem__read_file',
  'mcp__filesystem__read_media_file',
  'mcp__filesystem__read_multiple_file

## 4. Inspect first-pass governance overlay for filesystem MCP tools


In [5]:
def inspect_tool(name):
    tool = registry.get(name)
    return {
        'name': name,
        'side_effect_class': tool.side_effect_class,
        'requires_approval': tool.requires_approval,
        'governance_policy_group': tool.governance_policy_group,
        'environment_check_kind': tool.environment_check_kind,
    }

{
    'read_file': inspect_tool('mcp__filesystem__read_file'),
    'write_file': inspect_tool('mcp__filesystem__write_file'),
    'list_allowed_directories': inspect_tool('mcp__filesystem__list_allowed_directories'),
}

{'read_file': {'name': 'mcp__filesystem__read_file',
  'side_effect_class': 'safe',
  'requires_approval': False,
  'governance_policy_group': 'system_environment',
  'environment_check_kind': 'path_exists'},
 'write_file': {'name': 'mcp__filesystem__write_file',
  'side_effect_class': 'write',
  'requires_approval': True,
  'governance_policy_group': 'permission_authority',
  'environment_check_kind': 'none'},
 'list_allowed_directories': {'name': 'mcp__filesystem__list_allowed_directories',
  'side_effect_class': 'safe',
  'requires_approval': False,
  'governance_policy_group': 'system_environment',
  'environment_check_kind': 'none'}}

## 5. Direct wrapped invocation through a real filesystem MCP tool


In [6]:
wrapped = registry.get('mcp__filesystem__list_allowed_directories')
result = await wrapped.client.async_call_tool(wrapped.descriptor.original_name, {})
{'ok': result.ok, 'content': result.content, 'raw_result': result.data['raw_result']}

{'ok': True,
 'content': 'Allowed directories:\n/Volumes/2TB/MAS/openclaw-core/ORBIT',
 'raw_result': {'meta': None,
  'content': [{'type': 'text',
    'text': 'Allowed directories:\n/Volumes/2TB/MAS/openclaw-core/ORBIT',
    'annotations': None,
    'meta': None}],
  'structuredContent': {'content': 'Allowed directories:\n/Volumes/2TB/MAS/openclaw-core/ORBIT'},
  'isError': False}}

## 6. Governed runtime path: wrapped MCP `write_file` enters approval


In [7]:
class WriteBackend:
    def plan_from_messages(self, messages, session=None):
        return ExecutionPlan(
            source_backend='stub',
            plan_label='mcp-write-governed-showcase',
            tool_request=ToolRequest(
                tool_name='mcp__filesystem__write_file',
                input_payload={'path': 'tmp_mcp_write_showcase.txt', 'content': 'hello from mcp write showcase'},
                side_effect_class='write',
                requires_approval=True,
            ),
            should_finish_after_tool=False,
        )

db = Path('/tmp/orbit_mcp_write_showcase.db')
db.unlink(missing_ok=True)
store = SQLiteStore(db_path=db)
manager = SessionManager(store=store, backend=WriteBackend(), workspace_root=str(ROOT))
await async_register_mcp_server_tools(registry=manager.tool_registry, bootstrap=bootstrap)
session = manager.create_session(backend_name='stub', model='stub-model')
plan = manager.run_session_turn(session_id=session.session_id, user_input='please write through filesystem mcp')
{'plan_label': plan.plan_label, 'tool_request': plan.tool_request.tool_name if plan.tool_request else None, 'open_approvals': manager.list_open_session_approvals()}

{'plan_label': 'mcp-write-governed-showcase-waiting-for-approval',
 'tool_request': 'mcp__filesystem__write_file',
 'open_approvals': [{'session_id': 'session_352bdf932e73',
   'conversation_id': 'conversation:stub:1775165998',
   'approval_request_id': 'approval_8a2ec686f367',
   'tool_request': {'tool_name': 'mcp__filesystem__write_file',
    'input_payload': {'path': 'tmp_mcp_write_showcase.txt',
     'content': 'hello from mcp write showcase'},
    'requires_approval': True,
    'side_effect_class': 'write'},
   'source_backend': 'stub',
   'plan_label': 'mcp-write-governed-showcase',
   'opened_at': '2026-04-02T21:39:58.620969+00:00'}]}

## 7. Governed runtime path: wrapped MCP `read_file` under `system_environment`


In [8]:
existing = ROOT / 'tmp_mcp_read_showcase_existing.txt'
existing.write_text('hello from mcp governed read showcase')

class ReadBackend:
    def __init__(self, request):
        self.request = request
    def plan_from_messages(self, messages, session=None):
        return ExecutionPlan(
            source_backend='stub',
            plan_label='mcp-read-governed-showcase',
            tool_request=self.request,
            should_finish_after_tool=False,
        )

async def run_read_case(label, request):
    db = Path(f'/tmp/orbit_mcp_read_showcase_{label}.db')
    db.unlink(missing_ok=True)
    store = SQLiteStore(db_path=db)
    manager = SessionManager(store=store, backend=ReadBackend(request), workspace_root=str(ROOT))
    await async_register_mcp_server_tools(registry=manager.tool_registry, bootstrap=bootstrap)
    session = manager.create_session(backend_name='stub', model='stub-model')
    plan = manager.run_session_turn(session_id=session.session_id, user_input=label)
    messages = manager.list_messages(session.session_id)
    return {
        'plan_label': plan.plan_label,
        'final_text': plan.final_text,
        'last_kind': messages[-1].metadata.get('message_kind') if messages else None,
        'messages': [(m.turn_index, str(m.role), m.metadata.get('message_kind'), m.content) for m in messages],
    }

read_cases = {
    'absolute_existing': await run_read_case('absolute_existing', ToolRequest(tool_name='mcp__filesystem__read_file', input_payload={'path': str(existing)}, side_effect_class='safe', requires_approval=False)),
    'missing': await run_read_case('missing', ToolRequest(tool_name='mcp__filesystem__read_file', input_payload={'path': 'tmp_mcp_read_showcase_missing.txt'}, side_effect_class='safe', requires_approval=False)),
    'unknown': await run_read_case('unknown', ToolRequest(tool_name='mcp__filesystem__read_file', input_payload={}, side_effect_class='safe', requires_approval=False)),
}
read_cases

{'absolute_existing': {'plan_label': 'mcp-read-governed-showcase',
  'final_text': None,
  'last_kind': 'tool_result',
  'messages': [(1, 'user', None, 'absolute_existing'),
   (2,
    'tool',
    'tool_result',
    'Tool result from mcp__filesystem__read_file:\nhello from mcp governed read showcase')]},
 'missing': {'plan_label': 'mcp-read-governed-showcase-deny',
  'final_text': 'Current environment conditions do not allow mcp__filesystem__read_file.',
  'last_kind': 'policy_decision',
  'messages': [(1, 'user', None, 'missing'),
   (2,
    'assistant',
    'policy_decision',
    'Current environment conditions do not allow mcp__filesystem__read_file.')]},
 'unknown': {'plan_label': 'mcp-read-governed-showcase-recheck_environment',
  'final_text': 'The current environment state for mcp__filesystem__read_file is not fresh enough; recheck environment conditions before proceeding.',
  'last_kind': 'policy_decision',
  'messages': [(1, 'user', None, 'unknown'),
   (2,
    'assistant',
  

## 8. Takeaways

This first MCP slice now demonstrates:
- real stdio MCP server compatibility
- official Python `mcp` SDK-backed `list_tools()` and `call_tool()`
- real MCP tool registration into ORBIT ToolRegistry
- first-pass differentiated governance overlay for filesystem MCP tools
- real wrapped MCP tools entering ORBIT governed runtime paths

Current remaining caveat:
- relative-path semantics for filesystem MCP reads require careful alignment with server-side allowed-directory/path-resolution behavior; absolute-path cases already work cleanly.
